In [58]:
# Load the data and build the search index:

from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)


In [59]:
# Set up the OpenAI client:

import os
from dotenv import load_dotenv
load_dotenv()


from openai import OpenAI
openai_client = OpenAI(api_key=os.getenv("OPEN_API_KEY"))


In [60]:
# Create the assistant:

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [61]:
# This works fine. The search finds relevant FAQ entries about Ollama, and the LLM gives a good answer.

assistant.rag("How do I run Ollama locally?")

'To run Ollama locally, follow these steps:\n\n1. Install Ollama for your operating system:\n   - **macOS**: Download the `.pkg` installer from https://ollama.com/download and install it.\n   - **Windows**: Download the `.msi` installer from the same link and install it.\n   - **Linux**: Run this command in the terminal:\n     ```\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. After installation, open your terminal and start the local model by running:\n   ```\n   ollama run llama3\n   ```\n   This command will download the LLaMA 3 model (~4GB), start it locally, and open a chat-like interface for you to type questions.\n\n3. To verify the local Ollama server is running, execute:\n   ```\n   curl http://localhost:11434\n   ```\n   You should get a JSON response listing available models.\n\n4. To interact with Ollama programmatically in Python, install the Python client:\n   ```\n   pip install ollama\n   ```\n   Then use a minimal example like:\n   ```python\n   im

In [62]:
# The word "Olama" doesn't match "Ollama" in our index. We use lexical search, so it looks for the exact word and finds nothing. 
# The LLM gets these bad results and either says "I don't know" or answers with irrelevant information.

assistant.rag("How do I run Olama locally?")

'You can run Olama locally by serving the model on your own machine without making external API calls. This requires setting up Python and other necessary tools. Olama, like other local model-serving options (e.g., vLLM, LM Studio), typically exposes an OpenAI-compatible endpoint. This means you can run the course code locally by configuring it to point to the Olama local endpoint via the `base_url` and API key settings.\n\nIf you choose this approach, make sure your local environment is properly set up and documented so it remains reproducible. Running locally avoids regional blocks and paid API keys, giving you full control over the model serving environment.'

- This is the limitation of a fixed pipeline. 
- The search runs once with the exact query the user typed, and there's no second chance. The pipeline doesn't know the search failed, 
- so it can't try again with a corrected query.

- We need something smarter. We need an agent.


```mermaid
flowchart TD
    U([User: How do I run Olama?])
    S[search - Olama - no useful results]
    A([LLM: I don't have information about Olama.])

    U --> S --> A
```

### The agent alternative:

- An agent puts the LLM in charge.
- Instead of running search ourselves, we give the LLM a `search` tool. It decides when to call it and what to search for.


The same typo question now goes like this:

```mermaid
flowchart TD
    U([User: How do I run Olama?])
    L1[LLM: I'll search for 'Olama']
    S1[search - Olama - no useful results]
    L2[LLM: Hmm, no results. Maybe a typo for 'Ollama'?]
    S2[search - Ollama - found results!]
    A([LLM: Here's how to run Ollama locally...])

    U --> L1 --> S1 --> L2 --> S2 --> A
```

The LLM searched, saw the results were bad, and decided to try again with a different query.
It made that decision on its own. We didn't write any code to handle typos.

The difference is about who makes the decision:
  - With RAG, the developer decides. We fix the steps up front, so search always runs once with the exact user query.
  - With an agent, the LLM decides. It chooses which actions to take and when to stop

The mechanism that makes this possible is function calling, and that's what the rest of this lesson is about.

In [63]:
## Asking without any tools:

# First, let's see what the LLM does without any tools. We ask it a course-specific questions and look at the answer

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-4.1-mini",
    input=messages
)

response.output_text


# The model answers from its general knowledge. It doesn't know about our FAQ, so the answer is vague and not helpful.
# This is exactly why we need RAG, and why we want to hand the model a tool

"I'd be happy to help! Could you please tell me which course you are referring to?"

### Defining the tool

- First we define a top-level `search` function that queries the `index` directly.
- The model will reference it by this name. We keep the Python function and the tool name aligned, so the dispatch is easier later

In [64]:


def search(query):
        boost_dict={"question": 2.0, "section": 0.5} # Boost the question field twice as much as the section field
        filter_dict={"course": "llm-zoomcamp"} # Filter results to only include documents from the specified course

        return index.search(
            query, 
            boost_dict=boost_dict,
            filter_dict=filter_dict,
            num_results=5
        )

Next we tell the model about this function.
The model doesn't see our Python code, it sees only a schema describing what the function does and what arguments it takes.

In [65]:


search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

# Description - Is the most important field, because the model reads it to decide when to call the function
# parameters - is a JSON schema for the arguments
# Required - We make query as required

### Sending the question with the tool

Now we send the same question as before, but this time we include the tool in the request:

In [67]:


response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

# Look at the response. Instead of a message with the answer, the response contains a 'function call' entry.
# The model decided it needs to search the FAQ before answering. Rather than reply, it asked us to run the search function first

# Look at the arguments too. The model didn't pass our question verbatim (Exactly as it was spoken or written)
# It judged the raw question wasn't the best query to search with. So it rewrote our enrollment question into search keywords like "enroll late join course".

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment eligibility new student"}', call_id='call_vMIyXzGoYTeARMJHeN7nJuMp', name='search', type='function_call', id='fc_03eb94c123ae43ee006a446b81b0608197b8603bedaefd1154', namespace=None, status='completed')]

### Executing the function and sending the result back

The function call contains JSON arguments. We parse them, call our `search` function, and serialize the result.

In [68]:


import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

# Now we send this result back to the model
# First, we add the model's output to the conversation history - the model needs to see its own function call. Then we add the tool result

messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

# The call_id links the tool result to the specific function call the model requested. If the model makes multiple function calls in one turn, each one gets its own call_id.

messages


[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment eligibility new student"}', call_id='call_vMIyXzGoYTeARMJHeN7nJuMp', name='search', type='function_call', id='fc_03eb94c123ae43ee006a446b81b0608197b8603bedaefd1154', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_vMIyXzGoYTeARMJHeN7nJuMp',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "1e829a8c6f",\n    "course": "llm-zoomcamp",\n    "section": "Module 2: Vector Search",\n    "question": "Do I need a new GitHub repo for Module 2, or just a new codespace?"

### Asking the model again

We call the API a second time with the expanded history:

In [69]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

response.output_text



'Yes, you can still join.\n\nAccording to the course FAQ, you can start learning even if you discovered it late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.'


- This time the model has the original question, its own decision to call search, and the FAQ results. It can now produce a proper course-specific answer.

- We have to send the whole history because LLMs are stateless between API calls. The memory is the list you send as input. 
- If you send only the tool result, the model has no idea what's going on. So on this second call we replay everything we have so far. 
- That means the question, the decision to call search, and the result we got back.

- That's the full function-calling loop for a single turn. With plain RAG we made one call, and here we make two. Turning RAG agentic means more round-trips.

- People call this pattern "agentic RAG", "tool use", or "function calling". The idea behind all of them is the same. The LLM decides which tools to call.

### Token usage and cost

In [70]:


usage = response.usage
usage.input_tokens, usage.output_tokens

# For each model the provider publishes a price per million input tokens and per million output tokens. Plug those numbers in to convert tokens to dollars.


def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))


Total cost: $ 0.0001176


In [71]:
# gpt-4.1-mini

input_price = 0.40 / 1_000_000  # $0.40 per 1M tokens for gpt-4.1-mini
output_price = 1.60 / 1_000_000  # $1.60 per 1M tokens for gpt-4.1-mini

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00044800000000000005

In the above session, we did function calling by hand - We sent a message and got back a function call, we ran it, sent the result back, and got the answer

That works for one function call. It breaks down when the model wants to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we need a loop that keeps calling the model and running tools until it's done

An agent is exactly that



### Anatomy of an agent
---

With the LLM in the driver's seat, we have an agent. It's an AI Assistant whose goal is to help the user.

An agent has three parts:

- **Instructions** - the role and behavior we want. We pass this as the `developer`/ `system`message. The better the instructions, the better the agent helps
- **Tools** - the functions the agent can call to carry out the task. For us that's only `search`
- **Memory** - the message history. We append every prompt, every model output, and every tool result. The agent reads this to know what it has already tried

### A developer / system prompt
---

So far we've relied on the model to figure out when to search. We make that more reliable with a `developer`/ `system`message that spells out how to behave.
This is where we give the agent its role. The same message also pushes it toward multiple searches, so we get to watch the loop run more than once

In [72]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

### A fucntion-call helper
---

We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper.
It turns the JSON arguments into a Python dict, calls the right function, and serializes the result.
We only have one tool for now, so we dispatch on the function name directly

In [74]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent = 2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

### Processing one response
---

Let's process a single model response. We append each output entry to the conversation, print any messages, and run any function calls.

Function-call results get appended too

The `has_function_calls` flag tells us whether the model needs another API call. If the response contains a function call, the updated `messages` has tool output the model hasn't seen yet. We'll need to send it back.

In [75]:
question = "I just discovered the course. Can I join it ?"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

print("\n\n---\n\n", messages)

function_call: search {"query":"join course discovered course can I join"}
function_call: search {"query":"course enrollment discovered course join"}
function_call: search {"query":"late enrollment course join after start"}


---

 [{'role': 'system', 'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."}, {'role': 'user', 'content': 'I just discovered the course. Can I join it ?'}, ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join"}', call_id='call_LMCcdebhSsvpbHwwVJmIWt4M', name='search', type='function_call', id='fc_0654a7129b955b

### The full agent loop
---

We wrap this in a `while` loop. The loop keeps calling the model until it returns a response without any function calls.

We also keep an iteration counter so we can see how many round-trips happened

In [79]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if not has_function_calls:
        break




# This is the core agent loop. The model reasons about the next action. Your code performs it, and the model sees the result on the next turn. 
# The loop stops when the model returns a final answer with no more tool calls.

# We don't decide how many times the model searches. The model does, and we keep looping until it stops asking for tools.

# The exit condition is the simplest one possible. No function calls this turn means we're done. 
# Other frameworks add safety nets on top, like a max iteration count, a token budget, or a wall-clock limit. 
# You might cap it at five iterations and force an answer on the last one. The core is still this one flag.

iteration #1...
ASSISTANT:
Yes — you can still join the course.

A few key points:
- You can start anytime.
- If you want a certificate, you need to submit your project while submissions are still open.
- Homework forms close too, so once they’re closed, there are no late submissions.

If you want, I can also help you get started with the course materials. Are there other areas you’d like to explore?


### Wrap it in a function
---

Let's wrap the loop in a function so we can reuse it. The function takes
- instructions
- question as parameter

and return the final answer

In [80]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if not has_function_calls:
            break

    return last_answer

In [81]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local Ollama FAQ"}
function_call: search {"query":"Ollama run locally install macOS Windows Linux FAQ"}
function_call: search {"query":"local Ollama setup model run server FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. Install it from: https://ollama.com/download  
   - macOS: download the `.pkg`
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Start a model:
   ```bash
   ollama run llama3
   ```
   This downloads the model and starts a local chat session.

3. Verify the local server is running:
   ```bash
   curl http://localhost:11434
   ```

4. If you want to use it from Python:
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['message']['content'])
   ```

If y

'To run Ollama locally:\n\n1. Install it from: https://ollama.com/download  \n   - macOS: download the `.pkg`\n   - Windows: download the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Start a model:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and starts a local chat session.\n\n3. Verify the local server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. If you want to use it from Python:\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection refused error, restart the server with:\n```bash\nollama serve\n```\nor, in a notebook:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you want, I can also help with choosing a smaller model, using Oll

In [82]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered the course can I still join"}
function_call: search {"query":"course FAQ enrollment late join discovered course"}
function_call: search {"query":"can I still join the course after it started FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If your goal is to get a certificate, make sure you submit your project while submissions are still being accepted.

If you want, I can also help you figure out the best way to catch up quickly or explain the certificate requirements.


'Yes — you can still join the course.\n\nIf your goal is to get a certificate, make sure you submit your project while submissions are still being accepted.\n\nIf you want, I can also help you figure out the best way to catch up quickly or explain the certificate requirements.'

### Encouraging multiple searches
---

The model often answers after the first search, even when more searches would help. It reasons that it already knows enough. We push it to explore more by rewriting the instructions

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

# Now the agent makes multiple searches per question and doesn't stop after the first round of results. 
# The instructions are how we steer the agent. It can still decide to skip ahead sometimes, so don't expect it to follow them every single run.

iteration #1...
function_call: search {"query":"join the course enroll late discovered course can I join"}
iteration #2...
function_call: search {"query":"can I join course after it started certificate project submissions still accepting submissions late join self-paced"}
iteration #3...
ASSISTANT:
Yes — you can still join the course even if you just discovered it.

If you want a certificate, make sure you submit your project while submissions are still being accepted.

If you want, I can also explain the difference between joining self-paced vs. joining a live cohort. Are there other areas you’d like to explore?


'Yes — you can still join the course even if you just discovered it.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted.\n\nIf you want, I can also explain the difference between joining self-paced vs. joining a live cohort. Are there other areas you’d like to explore?'

### Restricting off-topic questions
---

Right now the agent will answer anything. Ask it about chess and it will try

In [85]:
agent_loop(instructions, "What's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit course FAQ"}
function_call: search {"query":"what is queen gambit queen's gambit opening FAQ"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening FAQ"}
iteration #3...
ASSISTANT:
A **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the c-pawn to try to gain control of the center and pull Black’s d-pawn away. It’s one of the most famous and solid openings in chess.

There are two main types:
- **Queen’s Gambit Accepted**: Black takes the c-pawn.
- **Queen’s Gambit Declined**: Black does not take it.

If you want, I can also explain:
- the basic ideas behind it,
- the main variations,
- or how to play it as White or defend against it as Black.

Are there other areas you want to explore?


'A **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn to try to gain control of the center and pull Black’s d-pawn away. It’s one of the most famous and solid openings in chess.\n\nThere are two main types:\n- **Queen’s Gambit Accepted**: Black takes the c-pawn.\n- **Queen’s Gambit Declined**: Black does not take it.\n\nIf you want, I can also explain:\n- the basic ideas behind it,\n- the main variations,\n- or how to play it as White or defend against it as Black.\n\nAre there other areas you want to explore?'

We want a course assistant, not a general chatbot. We tighten the instructions so the agent only answers from the FAQ. For our own use we might be fine letting it answer from general knowledge. So treat this mainly as an illustration of steering scope.

In [86]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening queen gambit"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen’s gambit,” so I can’t answer that from the course materials.

If you meant something else course-related, feel free to rephrase with more context. Are there other areas you want to explore?


'I couldn’t find a course FAQ entry for “queen’s gambit,” so I can’t answer that from the course materials.\n\nIf you meant something else course-related, feel free to rephrase with more context. Are there other areas you want to explore?'